# Dynamic Narration Graph — Dataset Visualisation

Exploratory analysis and figures for the seven corpora behind GNSM: **LitBank,
PDNC, BookCoref, EvolvTrip/LitCharToM, ConStory-Bench, FABLES, GPT4-Books**.

---

## ⚠️ Load the datasets before running this notebook

This notebook contains **analysis only** — it downloads nothing. Run Steps 1–6 of
[`README.md`](README.md) first (clone the six git corpora, pull the two Hugging
Face corpora, unzip FABLES, download BookCoref).

**No models are required.** Nothing here runs a forward pass, so the 72 GB in
`models/` can be skipped entirely.

## How to run

1. **Set `ROOT`** in the Setup cell below if your repo is not at
   `/content/Dynamic-Narration-Graph`. That is the only path you ever need to edit.
2. **Run top to bottom, in order.** Cells share state across sections — `df`,
   `form_cat`, `ent_cats`, `pdnc`, `as_list`, `norm`, `bc`, `g`, `cur`, `ood`,
   `fab`, `leaf`, `HF`, `EV`, `pr`, `res`. After a runtime restart, start again
   from Setup.
3. Sections are independent *as groups*: if a corpus isn't downloaded, skip its
   whole section (but always run Setup first).

## What each section needs, and how slow it is

| Section | Needs | Notes |
| --- | --- | --- |
| LitBank | `data/litbank` | fast |
| PDNC | `data/pdnc` | fast |
| BookCoref | `data/bookcoref` | parses ~107 MB of JSONL |
| EvolvTrip | `data/evolvtrip_data` | fast |
| ConStory-Bench | `data/constory_bench` | **reads ~2.7 GB across 33 CSVs (~30 s)** |
| FABLES | `data/fables` | unzips `FABLES.json` if needed |
| GPT4-Books | `data/gpt4_books` | reads 1,142 files |

_Cells ship without stored outputs — run them to regenerate every figure._

In [ ]:
# ── SETUP ── run this first, and again after every runtime restart.
#
# Load the datasets BEFORE running this notebook (README.md, Steps 1-6).
# No models are needed anywhere below.

from pathlib import Path
import ast, collections, itertools, json, re, time, zipfile
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path("/content/Dynamic-Narration-Graph")   # <- the only line you edit
DATA = ROOT / "data"
assert DATA.is_dir(), f"data/ not found under {ROOT} - load the datasets first"

print("ROOT:", ROOT)
print("corpora present:", sorted(p.name for p in DATA.iterdir() if p.is_dir()))

# ---- shared plot theme, used by every figure in this notebook ----
SURFACE, INK, INK2 = "#fcfcfb", "#0b0b0b", "#52514e"
MUTED, GRID, BASE  = "#898781", "#e1e0d9", "#c3c2b7"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 9, "text.color": INK, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": BASE,
    "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 130,
})

def ink_on(hexc):
    """Readable label colour on top of a filled mark."""
    r, g, b = (int(hexc[i:i+2], 16) / 255 for i in (1, 3, 5))
    return INK if 0.2126*r + 0.7152*g + 0.0722*b > 0.5 else "#ffffff"

---

# 1 · LitBank

100 literary excerpts of ~2,100 tokens each, with four annotation layers:
nested entities (4 levels), event triggers, coreference, and quotations.

Needs `data/litbank`.

In [ ]:
# ── BLOCK 1 ── LitBank: structure probe
from pathlib import Path
import collections, itertools

LITBANK = DATA / "litbank"
assert LITBANK.is_dir(), f"not found: {LITBANK}"

print("=== annotation layers ===")
for p in sorted(LITBANK.iterdir()):
    if p.is_dir() and not p.name.startswith("."):
        files = [f for f in p.rglob("*") if f.is_file()]
        ext = collections.Counter(f.suffix or "<none>" for f in files)
        subs = sorted(c.name for c in p.iterdir() if c.is_dir())
        print(f"{p.name:12} files={len(files):5}  ext={dict(ext)}  subdirs={subs}")

print("\n=== sample head per layer ===")
for rel in ["entities/tsv", "events/tsv", "coref/conll", "coref/brat",
            "quotations", "original"]:
    d = LITBANK / rel
    if not d.is_dir():
        print(f"\n-- {rel:16} MISSING")
        continue
    files = sorted(f for f in d.iterdir() if f.is_file())
    print(f"\n-- {rel:16} {len(files)} files | e.g. {files[0].name}")
    with open(files[0], encoding="utf-8") as f:
        for line in itertools.islice(f, 6):
            print("   ", line.rstrip()[:130])

In [ ]:
# ── BLOCK 2 ── LitBank: per-document aggregates
from pathlib import Path
import collections, itertools, re
import pandas as pd

LITBANK = DATA / "litbank"

# --- A. what is actually inside the two dirs named "tsv" ---
for rel in ["coref/tsv", "quotations/tsv"]:
    d = LITBANK / rel
    files = sorted(f.name for f in d.iterdir() if f.is_file())
    print(f"-- {rel}: {len(files)} files | first: {files[:3]}")
    with open(d / files[0], encoding="utf-8") as f:
        for line in itertools.islice(f, 4):
            print("   ", line.rstrip()[:130])
    print()

# --- B. per-document aggregates ---
def rows_of(path):
    with open(path, encoding="utf-8") as f:
        return [ln.rstrip("\n").split("\t") for ln in f if ln.strip()]

docs = sorted(p.stem for p in (LITBANK / "entities" / "tsv").glob("*.tsv"))
ent_cats = collections.Counter()   # (level, category) across the corpus
ev_tags  = collections.Counter()
records  = []

for doc in docs:
    rec = {"doc": doc}

    toks = rows_of(LITBANK / "entities" / "tsv" / f"{doc}.tsv")
    rec["tokens"] = len(toks)
    for lvl in range(1, 5):
        cats = collections.Counter(
            t[lvl].split("-", 1)[1] for t in toks
            if len(t) > lvl and t[lvl].startswith("B-")
        )
        rec[f"ents_L{lvl}"] = sum(cats.values())
        for cat, n in cats.items():
            ent_cats[(lvl, cat)] += n

    ev = rows_of(LITBANK / "events" / "tsv" / f"{doc}.tsv")
    ev_tags.update(t[1] for t in ev if len(t) > 1)
    rec["events"] = sum(1 for t in ev if len(t) > 1 and t[1].startswith("B-"))

    clusters = collections.Counter()
    with open(LITBANK / "coref" / "conll" / f"{doc}.conll", encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            col = line.rstrip("\n").split("\t")[-1]
            for m in re.finditer(r"\((\d+)", col):
                clusters[int(m.group(1))] += 1
    rec["coref_clusters"]   = len(clusters)
    rec["coref_mentions"]   = sum(clusters.values())
    rec["coref_singletons"] = sum(1 for v in clusters.values() if v == 1)

    records.append(rec)

df = pd.DataFrame(records)
print("=== per-document table ===", df.shape)
print(df.head(5).to_string(index=False))
print("\n=== summary ===")
print(df.drop(columns=["doc"]).describe().round(1).to_string())
print("\n=== entity categories by nesting level ===")
for lvl in range(1, 5):
    at = {c: n for (l, c), n in ent_cats.items() if l == lvl}
    print(f"  L{lvl}: total={sum(at.values()):6}  {dict(sorted(at.items(), key=lambda x: -x[1]))}")
print("\n=== event tag vocabulary ===", dict(ev_tags))

> **Note on the `events` column above.** It comes out **all zeros** — the filter
> looks for `B-EVENT`, but LitBank tags triggers with a bare `EVENT`
> (7,849 corpus-wide). The next cell recomputes the column correctly. The
> mistake is kept because the `event tag vocabulary` print is what revealed the
> real schema.

In [ ]:
# ── BLOCK 3 ── LitBank: fix events, parse coref/tsv, probe quotations
from pathlib import Path
import collections, itertools
import pandas as pd

LITBANK = DATA / "litbank"

def rows_of(path):
    with open(path, encoding="utf-8") as f:
        return [ln.rstrip("\n").split("\t") for ln in f if ln.strip()]

docs = sorted(p.stem for p in (LITBANK / "entities" / "tsv").glob("*.tsv"))

# --- A. events: tag is bare "EVENT" ---
ev_per_doc = {}
for doc in docs:
    ev = rows_of(LITBANK / "events" / "tsv" / f"{doc}.tsv")
    ev_per_doc[doc] = sum(1 for t in ev if len(t) > 1 and t[1] != "O")
df["events"] = df["doc"].map(ev_per_doc)
print("events/doc:", df["events"].min(), "-", df["events"].max(),
      "| mean", round(df["events"].mean(), 1))

# --- B. coref/tsv: mention form x category, span lengths, cluster sizes ---
row_types  = collections.Counter()
form_cat   = collections.Counter()   # (FORM, CATEGORY)
span_len   = collections.Counter()   # tokens per mention (same-sentence only)
men_per_doc = {}

for doc in docs:
    rows = rows_of(LITBANK / "coref" / "tsv" / f"{doc}.ann")
    row_types.update(r[0] for r in rows)
    mentions = [r for r in rows if r[0] == "MENTION" and len(r) >= 9]
    men_per_doc[doc] = len(mentions)
    for r in mentions:
        form_cat[(r[8], r[7])] += 1
        if r[2] == r[4]:
            span_len[int(r[5]) - int(r[3])] += 1

df["coref_tsv_mentions"] = df["doc"].map(men_per_doc)

print("\n=== coref/tsv row types ===", dict(row_types))
print("\n=== mention FORM x CATEGORY ===")
forms = sorted({f for f, _ in form_cat})
cats  = sorted({c for _, c in form_cat}, key=lambda c: -sum(
    n for (f, cc), n in form_cat.items() if cc == c))
print(f"{'':6}" + "".join(f"{c:>8}" for c in cats) + f"{'TOTAL':>9}")
for f in forms:
    row = [form_cat.get((f, c), 0) for c in cats]
    print(f"{f:6}" + "".join(f"{v:>8}" for v in row) + f"{sum(row):>9}")

print("\n=== mention span length (tokens) ===")
tot = sum(span_len.values())
for k in sorted(span_len)[:8]:
    print(f"  {k+1:2} tok: {span_len[k]:6}  ({100*span_len[k]/tot:4.1f}%)")

# --- C. quotations: find a non-empty file ---
qdir = LITBANK / "quotations" / "tsv"
qfiles = sorted(qdir.glob("*.ann"))
nonempty = [f for f in qfiles if f.stat().st_size > 0]
print(f"\n=== quotations === {len(qfiles)} files, {len(nonempty)} non-empty")
qtypes = collections.Counter()
for f in nonempty:
    qtypes.update(r[0] for r in rows_of(f))
print("row types:", dict(qtypes))
if nonempty:
    print("sample:", nonempty[0].name)
    for line in itertools.islice(open(nonempty[0], encoding="utf-8"), 6):
        print("   ", line.rstrip()[:130])

In [ ]:
# ── BLOCK 4 ── LitBank: the charts   (run after blocks 2 and 3)
# Plot theme (SURFACE / INK / GRID / ink_on) lives in the Setup cell.
import collections, re
import matplotlib.pyplot as plt

CAT = {"PER": "#2a78d6", "FAC": "#eb6834", "LOC": "#1baf7a",
       "GPE": "#eda100", "VEH": "#e87ba4", "ORG": "#008300"}
ORDER = ["PER", "FAC", "LOC", "GPE", "VEH", "ORG"]

def stacked(ax, rows, counts, totals, title):
    """rows: labels top-to-bottom; counts: {(row, cat): n}"""
    ypos, left = list(range(len(rows)))[::-1], [0.0] * len(rows)
    for c in ORDER:
        vals = [100 * counts.get((r, c), 0) / totals[r] for r in rows]
        ax.barh(ypos, vals, left=left, color=CAT[c], height=0.6,
                edgecolor=SURFACE, linewidth=1.6, label=c, zorder=3)
        for yp, v, l in zip(ypos, vals, left):
            if v >= 5:
                ax.text(l + v/2, yp, f"{v:.0f}", ha="center", va="center",
                        fontsize=7.5, color=ink_on(CAT[c]))
        left = [l + v for l, v in zip(left, vals)]
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{r}\n{totals[r]:,}" for r in rows], fontsize=8.5, color=INK2)
    ax.set_xlim(0, 100); ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xticklabels(["0", "25", "50", "75", "100%"])
    ax.set_title(title, loc="left", fontsize=10, color=INK, pad=8)
    ax.spines["left"].set_visible(False); ax.tick_params(left=False)

fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.6))

# 1 - mention form
forms  = ["PRON", "NOM", "PROP"]
f_tot  = {f: sum(form_cat.get((f, c), 0) for c in ORDER) for f in forms}
stacked(axes[0, 0], forms, form_cat, f_tot,
        "Mention form by entity category  (share of mentions, %)")

# 2 - nesting depth
levels = [f"L{i}" for i in (1, 2, 3, 4)]
lvl_c  = {(f"L{l}", c): n for (l, c), n in ent_cats.items()}
l_tot  = {f"L{l}": sum(n for (ll, _), n in ent_cats.items() if ll == l) for l in (1, 2, 3, 4)}
stacked(axes[0, 1], levels, lvl_c, l_tot,
        "Entity category by nesting level  (share of mentions, %)")

# 3 - coref cluster sizes
sizes = collections.Counter()
for doc in docs:
    cl = collections.Counter()
    with open(LITBANK / "coref" / "conll" / f"{doc}.conll", encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            for m in re.finditer(r"\((\d+)", line.rstrip("\n").split("\t")[-1]):
                cl[int(m.group(1))] += 1
    sizes.update(cl.values())

bins = [("1", lambda s: s == 1), ("2", lambda s: s == 2), ("3", lambda s: s == 3),
        ("4", lambda s: s == 4), ("5", lambda s: s == 5),
        ("6-10", lambda s: 6 <= s <= 10), ("11-20", lambda s: 11 <= s <= 20),
        ("21+", lambda s: s >= 21)]
tot_cl = sum(sizes.values())
vals   = [sum(n for s, n in sizes.items() if fn(s)) for _, fn in bins]
ax = axes[1, 0]
ax.bar([b for b, _ in bins], vals, color="#2a78d6", width=0.68, zorder=3)
for i, v in enumerate(vals):
    ax.text(i, v + tot_cl*0.012, f"{100*v/tot_cl:.0f}%", ha="center",
            fontsize=8, color=INK2)
ax.set_ylim(0, max(vals) * 1.16)
ax.set_xlabel("mentions in cluster"); ax.set_ylabel("clusters")
ax.set_title(f"Coreference cluster size   ({tot_cl:,} clusters, 100 documents)",
             loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

# 4 - annotation density per document
ax = axes[1, 1]
x = 1000 * df["ents_L1"] / df["tokens"]
y = 1000 * df["events"]  / df["tokens"]
ax.scatter(x, y, s=40, color="#2a78d6", alpha=0.75,
           edgecolor=SURFACE, linewidth=1.1, zorder=3)
ax.set_xlabel("entity mentions per 1,000 tokens")
ax.set_ylabel("event triggers per 1,000 tokens")
ax.set_title("Annotation density  (1 dot = 1 document)", loc="left",
             fontsize=10, color=INK, pad=8)
ax.grid(color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
zero = df.loc[df["events"] == 0]
for _, r in zero.iterrows():
    ax.annotate(r["doc"][:26], (1000*r["ents_L1"]/r["tokens"], 0),
                textcoords="offset points", xytext=(6, 7),
                fontsize=7.5, color=INK2)

handles = [plt.Rectangle((0, 0), 1, 1, color=CAT[c]) for c in ORDER]
fig.legend(handles, ORDER, loc="lower center", ncol=6, frameon=False,
           bbox_to_anchor=(0.5, -0.005), fontsize=9, labelcolor=INK2)
fig.suptitle("LitBank - 100 literary excerpts, ~2,100 tokens each",
             x=0.008, ha="left", fontsize=12.5, color=INK)
fig.tight_layout(rect=[0, 0.045, 1, 0.955])
plt.show()

# table view (three fills sit under 3:1 contrast on this surface)
print("\nmention form x category (counts)")
print(f"{'':6}" + "".join(f"{c:>7}" for c in ORDER) + f"{'total':>8}")
for f in forms:
    print(f"{f:6}" + "".join(f"{form_cat.get((f,c),0):>7}" for c in ORDER)
          + f"{f_tot[f]:>8}")
print(f"\nsingletons: {100*vals[0]/tot_cl:.1f}% of clusters"
      f" | events/1k: {y.min():.1f}-{y.max():.1f}"
      f" | entities/1k: {x.min():.1f}-{x.max():.1f}"
      f" | corr(ents, events) = {x.corr(y):+.2f}")

In [ ]:
# ── BLOCK 5 ── LitBank: outlier check + the quotations layer
import collections
import matplotlib.pyplot as plt

# --- A. is the zero-event document really empty? ---
for doc in df.nsmallest(3, "events")["doc"]:
    tags = collections.Counter(t[1] for t in rows_of(LITBANK/"events"/"tsv"/f"{doc}.tsv") if len(t) > 1)
    print(f"{doc[:44]:46} {dict(tags)}")

# --- B. quotations: schema + per-document stats ---
qdir = LITBANK / "quotations" / "tsv"
print("\nATTRIB rows (schema we never saw):")
for f in sorted(qdir.glob("*.ann")):
    rows = [r for r in rows_of(f) if r[0] == "ATTRIB"]
    if rows:
        for r in rows[:3]:
            print("   ", "\t".join(r)[:130])
        break

q_per_doc, q_len, spanning = {}, [], 0
for f in sorted(qdir.glob("*.ann")):
    quotes = [r for r in rows_of(f) if r[0] == "QUOTE" and len(r) >= 7]
    q_per_doc[f.stem] = len(quotes)
    for r in quotes:
        if r[2] == r[4]:
            q_len.append(int(r[5]) - int(r[3]) + 1)
        else:
            spanning += 1

qs = pd.Series(q_per_doc)
print(f"\nquotes: {qs.sum():,} total | {(qs == 0).sum()} docs with none"
      f" | median {qs.median():.0f}, max {qs.max()} in a ~2k-token excerpt"
      f" | {spanning} multi-sentence quotes excluded from the length panel")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.6))

ax = axes[0]
ax.hist(qs.values, bins=range(0, int(qs.max()) + 6, 5), color="#2a78d6",
        edgecolor=SURFACE, linewidth=1.2, zorder=3)
ax.set_xlabel("quotes in document"); ax.set_ylabel("documents")
ax.set_title("Dialogue density  (100 documents)", loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
ax.annotate(f"{(qs == 0).sum()} documents\nwith no dialogue", (0, (qs == 0).sum()),
            textcoords="offset points", xytext=(18, 26), fontsize=8, color=INK2,
            arrowprops=dict(arrowstyle="-", color=BASE, linewidth=0.8))

ax = axes[1]
ax.hist(q_len, bins=range(1, 62, 3), color="#2a78d6",
        edgecolor=SURFACE, linewidth=1.2, zorder=3)
ax.set_xlabel("quote length (tokens)"); ax.set_ylabel("quotes")
ax.set_title(f"Quote length  ({len(q_len):,} single-sentence quotes)", loc="left",
             fontsize=10, color=INK, pad=8)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

fig.tight_layout()
plt.show()

---

# 2 · PDNC

28 full novels (2.33M words) with quotation-level annotation: speaker,
addressees, quote type, and character mentions *inside* each quote.

Needs `data/pdnc`.

In [ ]:
# ── BLOCK 6 ── PDNC: structure probe
from pathlib import Path
import collections
import pandas as pd

PDNC = DATA / "pdnc"

print("=== repo top level ===")
for p in sorted(PDNC.iterdir()):
    if p.name.startswith("."):
        continue
    n = sum(1 for _ in p.rglob("*") if _.is_file()) if p.is_dir() else 1
    print(f"  {p.name:26} {'dir' if p.is_dir() else 'file':4}  files={n}")

novels = sorted(d.name for d in (PDNC / "data").iterdir() if d.is_dir())
print(f"\n=== {len(novels)} novels ===")
print("  " + ", ".join(novels))

# are all novels laid out the same way?
layouts = collections.Counter()
for nv in novels:
    layouts[tuple(sorted(f.name for f in (PDNC / "data" / nv).iterdir() if f.is_file()))] += 1
print(f"\n=== distinct file layouts: {len(layouts)} ===")
for files, n in layouts.most_common():
    print(f"  {n:2} novels: {list(files)}")

nv = novels[0]
print(f"\n=== {nv}: file sizes ===")
for f in sorted((PDNC / "data" / nv).iterdir()):
    if f.is_file():
        print(f"  {f.name:30} {f.stat().st_size/1e3:9.1f} KB")

for f in sorted((PDNC / "data" / nv).glob("*.csv")):
    d = pd.read_csv(f, nrows=3)
    print(f"\n-- {f.name}  ({len(d.columns)} cols)")
    print("   cols:", list(d.columns))
    with pd.option_context("display.max_colwidth", 45, "display.width", 200):
        print(d.head(2).to_string(index=False))

txt = sorted((PDNC / "data" / nv).glob("*.txt"))
if txt:
    print(f"\n-- {txt[0].name} (first 300 chars)")
    print("   ", open(txt[0], encoding="utf-8").read(300).replace("\n", " / "))

In [ ]:
# ── BLOCK 7 ── PDNC: per-novel aggregates
import ast, collections
import pandas as pd
from pathlib import Path

PDNC = DATA / "pdnc"
novels = sorted(d.name for d in (PDNC / "data").iterdir() if d.is_dir())

def as_list(v):
    if isinstance(v, str) and v.strip():
        try:
            out = ast.literal_eval(v)
        except Exception:
            return []
        return list(out) if isinstance(out, (list, set, tuple)) else [out]
    return []

def flat2(v):                      # nested per-subquotation lists -> count
    return sum(len(s) if isinstance(s, (list, tuple, set)) else 1 for s in as_list(v))

qtype, gender, category = collections.Counter(), collections.Counter(), collections.Counter()
n_addr, alias_n, unknown_names = collections.Counter(), [], collections.Counter()
rows = []

for nv in novels:
    ch   = pd.read_csv(PDNC / "data" / nv / "character_info.csv")
    qu   = pd.read_csv(PDNC / "data" / nv / "quotation_info.csv")
    text = (PDNC / "data" / nv / "novel_text.txt").read_bytes()

    gender.update(ch["Gender"].fillna("?"))
    category.update(ch["Category"].fillna("?"))
    alias_n.extend(ch["Aliases"].map(as_list).map(len))

    qtype.update(qu["quoteType"].fillna("?"))
    addr = qu["addressees"].map(as_list)
    n_addr.update(addr.map(len))

    names = set(ch["Main Name"].dropna())
    spk   = qu["speaker"].fillna("<none>")
    unknown_names.update(spk[~spk.isin(names)])
    top   = spk.value_counts()

    rows.append({
        "novel":        nv[:26],
        "words":        len(text.split()),
        "characters":   len(ch),
        "quotes":       len(qu),
        "speakers":     int(spk.nunique()),
        "q_per_10k":    round(10_000 * len(qu) / max(len(text.split()), 1), 1),
        "explicit_%":   round(100 * (qu["quoteType"] == "Explicit").mean(), 1),
        "top_spkr_%":   round(100 * top.iloc[0] / len(qu), 1) if len(top) else 0.0,
        "mean_addr":    round(addr.map(len).mean(), 2),
        "q_words_med":  int(qu["quoteText"].fillna("").str.split().str.len().median()),
        "in_q_mentions": int(qu["mentionTextsList"].map(flat2).sum()),
    })

pdnc = pd.DataFrame(rows)
print("=== per-novel ===")
print(pdnc.to_string(index=False))
print("\n=== corpus totals ===")
print(f"  {pdnc['quotes'].sum():,} quotes | {pdnc['characters'].sum():,} characters"
      f" | {pdnc['words'].sum():,} words | {pdnc['in_q_mentions'].sum():,} in-quote mentions")
print("\nquoteType   :", dict(qtype.most_common()))
print("gender      :", dict(gender.most_common()))
print("category    :", dict(category.most_common()))
print("addressees/quote:", dict(sorted(n_addr.items())))
al = pd.Series(alias_n)
print(f"aliases/char: mean {al.mean():.1f}, median {al.median():.0f}, max {al.max()}")
print("\nspeakers not in character_info (top 10):", dict(unknown_names.most_common(10)))

In [ ]:
# ── BLOCK 8 ── PDNC: speaker-resolution audit + charts
# The speakers "missing" above are aliases - this folds Aliases into the join.
import ast, collections, re
import pandas as pd
import matplotlib.pyplot as plt

def norm(s):
    return re.sub(r"[^a-z ]", "", str(s).lower()).strip()

audit, ambiguous = [], collections.Counter()
for nv in novels:
    ch = pd.read_csv(PDNC / "data" / nv / "character_info.csv")
    qu = pd.read_csv(PDNC / "data" / nv / "quotation_info.csv")

    main = set(ch["Main Name"].dropna())
    alias2main = {}
    for _, r in ch.iterrows():
        for a in as_list(r["Aliases"]):
            alias2main.setdefault(str(a), set()).add(r["Main Name"])
    for a, owners in alias2main.items():
        if len(owners) > 1:
            ambiguous[f"{nv}:{a}"] = len(owners)
    norm_keys = {norm(x) for x in main} | {norm(a) for a in alias2main}

    spk   = qu["speaker"].fillna("<none>").astype(str)
    exact = spk.isin(main)
    alias = ~exact & spk.isin(alias2main.keys())
    fuzzy = ~exact & ~alias & spk.map(norm).isin(norm_keys)
    audit.append({"novel": nv[:26], "n": len(qu),
                  "exact": int(exact.sum()), "alias": int(alias.sum()),
                  "fuzzy": int(fuzzy.sum()),
                  "unresolved": int((~(exact | alias | fuzzy)).sum())})

au = pd.DataFrame(audit)
tot = au[["exact", "alias", "fuzzy", "unresolved"]].sum()
print("=== speaker resolution, 37,131 quotes ===")
for k in ["exact", "alias", "fuzzy", "unresolved"]:
    print(f"  {k:11} {tot[k]:6,}  ({100*tot[k]/tot.sum():5.1f}%)")
print(f"\nambiguous aliases (one alias, several characters): {len(ambiguous)}")
print("  ", dict(list(ambiguous.items())[:6]))
print("\nnovels with any unresolved speaker:")
print(au[au.unresolved > 0][["novel", "n", "unresolved"]].to_string(index=False))

# ---------- charts ----------
STATUS = {"exact": "#0ca30c", "alias": "#fab219", "fuzzy": "#ec835a", "unresolved": "#d03b3b"}
QT     = {"Explicit": "#2a78d6", "Implicit": "#eb6834", "Anaphoric": "#1baf7a"}

qt_rows = []
for nv in novels:
    qu = pd.read_csv(PDNC / "data" / nv / "quotation_info.csv")
    c  = qu["quoteType"].fillna("?").value_counts()
    qt_rows.append({"novel": nv[:26], **{k: 100*c.get(k, 0)/len(qu) for k in QT}})
qt = pd.DataFrame(qt_rows).sort_values("Explicit")

fig, axes = plt.subplots(2, 2, figsize=(13, 9.2),
                         gridspec_kw={"height_ratios": [1.55, 1]})

def hstack(ax, frame, cols, colors, title, labels=True):
    y, left = range(len(frame)), [0.0] * len(frame)
    for c in cols:
        v = frame[c].values
        ax.barh(list(y), v, left=left, color=colors[c], height=0.68,
                edgecolor=SURFACE, linewidth=1.1, label=c, zorder=3)
        left = [l + x for l, x in zip(left, v)]
    ax.set_yticks(list(y))
    ax.set_yticklabels(frame["novel"] if labels else [""] * len(frame), fontsize=6.6, color=INK2)
    ax.set_xlim(0, 100); ax.set_xticks([0, 50, 100]); ax.set_xticklabels(["0", "50", "100%"])
    ax.set_title(title, loc="left", fontsize=10, color=INK, pad=8)
    ax.spines["left"].set_visible(False); ax.tick_params(left=False)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.13), ncol=4,
              frameon=False, fontsize=8, labelcolor=INK2)

hstack(axes[0, 0], qt, list(QT), QT, "Quote type by novel  (%)")
aup = au.copy()
for c in ["exact", "alias", "fuzzy", "unresolved"]:
    aup[c] = 100 * aup[c] / aup["n"]
hstack(axes[0, 1], aup.set_index("novel").loc[qt["novel"]].reset_index(),
       list(STATUS), STATUS, "Speaker name resolves against character_info  (%)", labels=False)

ax = axes[1, 0]
keys = ["0", "1", "2", "3", "4", "5", "6-10", "11+"]
vals = [n_addr.get(0, 0), n_addr.get(1, 0), n_addr.get(2, 0), n_addr.get(3, 0),
        n_addr.get(4, 0), n_addr.get(5, 0),
        sum(v for k, v in n_addr.items() if 6 <= k <= 10),
        sum(v for k, v in n_addr.items() if k >= 11)]
ax.bar(keys, vals, color="#2a78d6", width=0.68, zorder=3)
t = sum(vals)
for i, v in enumerate(vals):
    ax.text(i, v + t*0.012, f"{100*v/t:.0f}%", ha="center", fontsize=8, color=INK2)
ax.set_ylim(0, max(vals)*1.15); ax.set_xlabel("addressees on a quote"); ax.set_ylabel("quotes")
ax.set_title("Who is being spoken to  (37,131 quotes)", loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

ax = axes[1, 1]
ax.scatter(pdnc["q_per_10k"], pdnc["q_words_med"], s=44, color="#2a78d6",
           alpha=0.75, edgecolor=SURFACE, linewidth=1.1, zorder=3)
for _, r in pdnc.iterrows():
    if r["q_per_10k"] > 300 or r["q_words_med"] >= 29 or r["q_per_10k"] < 70:
        ax.annotate(r["novel"], (r["q_per_10k"], r["q_words_med"]),
                    textcoords="offset points", xytext=(6, 5), fontsize=7, color=INK2)
ax.set_xlabel("quotes per 10,000 words"); ax.set_ylabel("median quote length (words)")
ax.set_title("Dialogue style  (1 dot = 1 novel)", loc="left", fontsize=10, color=INK, pad=8)
ax.grid(color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

fig.suptitle("PDNC - 28 novels, 37,131 quotations, 1,228 characters",
             x=0.008, ha="left", fontsize=12.5, color=INK)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# ── BLOCK 9 ── PDNC: addressees, in-quote mentions, dialogue share
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def flat_all(v):
    out = []
    def rec(x):
        if isinstance(x, (list, tuple, set)):
            for i in x:
                rec(i)
        elif x is not None and str(x).strip():
            out.append(str(x))
    rec(as_list(v))
    return out

addr_state = collections.Counter()
ment_state = collections.Counter()
per_quote_mentions = collections.Counter()
share_rows, cum_curves, cover80 = [], [], []

for nv in novels:
    ch = pd.read_csv(PDNC / "data" / nv / "character_info.csv")
    qu = pd.read_csv(PDNC / "data" / nv / "quotation_info.csv")
    nbytes = len((PDNC / "data" / nv / "novel_text.txt").read_bytes())

    main = set(ch["Main Name"].dropna())
    aliases = {str(a) for _, r in ch.iterrows() for a in as_list(r["Aliases"])}
    norm_keys = {norm(x) for x in main | aliases}

    def state(name):
        if name in main:      return "exact"
        if name in aliases:   return "alias"
        if norm(name) in norm_keys: return "fuzzy"
        return "unresolved"

    for v in qu["addressees"]:
        for a in as_list(v):
            addr_state[state(str(a))] += 1
    for v in qu["mentionEntitiesList"]:
        names = flat_all(v)
        per_quote_mentions[min(len(names), 3)] += 1
        for m in names:
            ment_state[state(m)] += 1

    spans = qu["quoteByteSpans"].map(as_list)
    spoken = sum(int(e) - int(s) for row in spans for s, e in
                 (p for p in row if isinstance(p, (list, tuple)) and len(p) == 2))
    share_rows.append({"novel": nv[:26], "share": 100 * spoken / nbytes})

    counts = qu["speaker"].value_counts().values
    cum = np.cumsum(counts) / counts.sum() * 100
    cum_curves.append(np.pad(cum[:20], (0, max(0, 20 - len(cum))), constant_values=100))
    cover80.append(int(np.searchsorted(cum, 80) + 1))

def pct(c):
    t = sum(c.values())
    return {k: (v, 100*v/t) for k, v in c.most_common()}, t

a_pct, a_tot = pct(addr_state)
m_pct, m_tot = pct(ment_state)
print(f"=== addressee slots: {a_tot:,} ===")
for k, (v, p) in a_pct.items(): print(f"  {k:11} {v:7,}  ({p:5.1f}%)")
print(f"\n=== in-quote mention entities: {m_tot:,} ===")
for k, (v, p) in m_pct.items(): print(f"  {k:11} {v:7,}  ({p:5.1f}%)")
print(f"\nspeakers covering 80% of a novel's dialogue: median {int(np.median(cover80))}"
      f"  (range {min(cover80)}-{max(cover80)})")

sh = pd.DataFrame(share_rows).sort_values("share")
print(f"dialogue share of text: {sh['share'].min():.0f}%-{sh['share'].max():.0f}%"
      f", median {sh['share'].median():.0f}%")

# ---------- charts ----------
fig, axes = plt.subplots(2, 2, figsize=(13, 8.6), gridspec_kw={"height_ratios": [1.5, 1]})

ax = axes[0, 0]
ax.barh(range(len(sh)), sh["share"], color="#2a78d6", height=0.7, zorder=3)
ax.set_yticks(range(len(sh))); ax.set_yticklabels(sh["novel"], fontsize=6.6, color=INK2)
ax.set_xlabel("% of novel bytes inside quotations")
ax.set_title("How much of the book is speech", loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="x", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)

ax = axes[0, 1]
curves = np.vstack(cum_curves); x = np.arange(1, 21)
ax.fill_between(x, np.percentile(curves, 25, axis=0), np.percentile(curves, 75, axis=0),
                color="#2a78d6", alpha=0.18, linewidth=0, zorder=2)
ax.plot(x, np.median(curves, axis=0), color="#2a78d6", linewidth=2, zorder=3)
ax.axhline(80, color=BASE, linewidth=1, linestyle=(0, (4, 3)), zorder=1)
ax.text(20, 81.5, "80% of dialogue", ha="right", fontsize=7.5, color=INK2)
ax.set_xlim(1, 20); ax.set_ylim(0, 101); ax.set_xticks([1, 5, 10, 15, 20])
ax.set_xlabel("speaker rank within novel"); ax.set_ylabel("cumulative % of quotes")
ax.set_title("Dialogue concentrates on few characters  (median, IQR band)",
             loc="left", fontsize=10, color=INK, pad=8)
ax.grid(color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

ax = axes[1, 0]
keys = ["0", "1", "2", "3+"]
vals = [per_quote_mentions.get(i, 0) for i in range(4)]
ax.bar(keys, vals, color="#2a78d6", width=0.66, zorder=3)
t = sum(vals)
for i, v in enumerate(vals):
    ax.text(i, v + t*0.012, f"{100*v/t:.0f}%", ha="center", fontsize=8, color=INK2)
ax.set_ylim(0, max(vals)*1.15)
ax.set_xlabel("characters named inside the quote"); ax.set_ylabel("quotes")
ax.set_title("In-quote character mentions", loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

ax = axes[1, 1]
STATUS = {"exact": "#0ca30c", "alias": "#fab219", "fuzzy": "#ec835a", "unresolved": "#d03b3b"}
rows = [("addressees", addr_state, a_tot), ("in-quote mentions", ment_state, m_tot)]
for yi, (lab, c, t) in enumerate(rows):
    left = 0.0
    for k in STATUS:
        v = 100 * c.get(k, 0) / t
        ax.barh([yi], [v], left=[left], color=STATUS[k], height=0.5,
                edgecolor=SURFACE, linewidth=1.1,
                label=k if yi == 0 else None, zorder=3)
        if v >= 6:
            ax.text(left + v/2, yi, f"{v:.0f}", ha="center", va="center",
                    fontsize=7.5, color=ink_on(STATUS[k]))
        left += v
ax.set_yticks([0, 1]); ax.set_yticklabels([f"{r[0]}\n{r[2]:,}" for r in rows],
                                          fontsize=8.5, color=INK2)
ax.set_xlim(0, 100); ax.set_xticks([0, 50, 100]); ax.set_xticklabels(["0", "50", "100%"])
ax.set_title("Do the other name fields resolve too?", loc="left", fontsize=10, color=INK, pad=8)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.3), ncol=4, frameon=False,
          fontsize=8, labelcolor=INK2)

fig.suptitle("PDNC - relational layers", x=0.008, ha="left", fontsize=12.5, color=INK)
fig.tight_layout(rect=[0, 0, 1, 0.955])
plt.show()

---

# 3 · BookCoref

53 complete books (11.1M tokens) with character-only coreference — the dataset
that actually tests long-range entity tracking.

Needs `data/bookcoref`. Block 11 parses ~107 MB of JSONL.

In [ ]:
# ── BLOCK 10 ── BookCoref: structure probe
import json
from pathlib import Path

BC = DATA / "bookcoref"
for s in ["train", "validation", "test"]:
    p = BC / f"{s}.jsonl"
    print(f"{s:11}", f"{p.stat().st_size/1e6:8.1f} MB" if p.exists() else "MISSING")

row = json.loads(open(BC / "train.jsonl", encoding="utf-8").readline())
print("\nkeys:", list(row))
for k, v in row.items():
    print(f"  {k:14} {type(v).__name__:5} len={len(v) if hasattr(v, '__len__') else '-'}")

sents = row["sentences"]
flat  = [t for s in sents for t in s]
print(f"\ndoc_key={row['doc_key']}  gutenberg_key={row['gutenberg_key']}")
print(f"sentences={len(sents):,}  tokens={len(flat):,}")
print("  sentences[0][:14]:", sents[0][:14])

cl = row["clusters"]
print(f"\nclusters: {len(cl)}")
print(f"  cluster[0]: type={type(cl[0]).__name__} len={len(cl[0])}")
print(f"  cluster[0][:4]: {cl[0][:4]}")
print(f"  cluster sizes: {[len(c) for c in cl]}")
# NOTE: characters[i] is a dict {name, mentions} carrying every span - printing
# several of them dumps megabytes, so only the names are shown.
print(f"  characters ({len(row['characters'])}): {[c['name'] for c in row['characters']][:8]}")

m = cl[0][0]
print(f"\nfirst mention of cluster 0: {m}")
try:
    print("  as flat-token span :", " ".join(flat[m[0]:m[1] + 1])[:80])
except Exception as e:
    print("  flat-token span failed:", type(e).__name__, e)

In [ ]:
# ── BLOCK 11 ── BookCoref: verify span semantics, then aggregate
import json, collections
import numpy as np, pandas as pd
from pathlib import Path

BC = DATA / "bookcoref"

# --- verification on the small split, bounded output ---
row  = json.loads(open(BC / "validation.jsonl", encoding="utf-8").readline())
flat = [t for s in row["sentences"] for t in s]
print("doc:", row["doc_key"], "| tokens:", f"{len(flat):,}")
print("character keys:", list(row["characters"][0]))
print("names:", [c["name"] for c in row["characters"]][:12])
print("\nfirst 3 sentences:", [" ".join(s) for s in row["sentences"][:3]])
print("\nspan decode (cluster 0):")
for a, b in row["clusters"][0][:6]:
    print(f"  [{a:6},{b:6}] -> {' '.join(flat[a:b+1])!r}")
same = [row["characters"][i]["mentions"] == row["clusters"][i] for i in range(len(row["clusters"]))]
print("\ncharacters[i].mentions == clusters[i] ?", all(same), f"({sum(same)}/{len(same)})")

# --- aggregate over all 53 books ---
recs, gaps, span_len = [], [], collections.Counter()
for split in ["train", "validation", "test"]:
    with open(BC / f"{split}.jsonl", encoding="utf-8") as f:
        for line in f:
            r     = json.loads(line)
            ntok  = sum(len(s) for s in r["sentences"])
            sizes = [len(c) for c in r["clusters"]]
            for c in r["clusters"]:
                starts = sorted(m[0] for m in c)
                if len(starts) > 1:
                    gaps.extend(np.diff(starts).tolist())
                for a, b in c:
                    span_len[min(b - a + 1, 6)] += 1
            recs.append({"split": split, "doc": r["doc_key"][:30], "tokens": ntok,
                         "sents": len(r["sentences"]), "chars": len(r["clusters"]),
                         "mentions": sum(sizes), "biggest": max(sizes), "smallest": min(sizes)})

bc = pd.DataFrame(recs)
bc["ment_per_1k"] = (1000 * bc["mentions"] / bc["tokens"]).round(1)
print("\n=== 53 books ===")
print(bc.groupby("split")[["tokens", "chars", "mentions", "biggest"]].agg(["count", "mean", "min", "max"]).round(0).to_string())
print("\ncorpus:", f"{bc['tokens'].sum():,} tokens |",
      f"{bc['mentions'].sum():,} mentions |", f"{bc['chars'].sum()} characters")
print("\nlongest / shortest books:")
print(bc.nlargest(3, "tokens")[["doc", "tokens", "chars", "mentions"]].to_string(index=False))
print(bc.nsmallest(3, "tokens")[["doc", "tokens", "chars", "mentions"]].to_string(index=False))

g = np.array(gaps)
print(f"\n=== gap between consecutive mentions of the same character ({len(g):,}) ===")
for p in [50, 75, 90, 95, 99, 99.9]:
    print(f"  p{p:<5} {np.percentile(g, p):8,.0f} tokens")
print(f"  max     {g.max():8,} tokens | over 4k apart: {100*(g > 4096).mean():.2f}%"
      f" | over 32k: {100*(g > 32768).mean():.3f}%")
print("\nmention span length:", {f"{k}{'+' if k == 6 else ''}": v for k, v in sorted(span_len.items())})

In [ ]:
# ── BLOCK 12 ── BookCoref: the charts
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 8.4))
WINDOWS = [(4_096, "4K"), (32_768, "32K"), (131_072, "128K")]

# A - book length against real context windows
ax = axes[0, 0]
s = bc.sort_values("tokens").reset_index(drop=True)
ax.scatter(s["tokens"], s.index + 1, s=26, color="#2a78d6",
           alpha=0.8, edgecolor=SURFACE, linewidth=0.8, zorder=3)
ax.set_xscale("log"); ax.set_xlim(2.5e4, 9e5); ax.set_ylim(0, 57)
for v, lab in WINDOWS:
    if not (2.5e4 <= v <= 9e5):
        continue                      # 4K sits left of the axis - skip it
    ax.axvline(v, color=BASE, linewidth=1, linestyle=(0, (4, 3)), zorder=1)
    ax.text(v, 54.5, lab, ha="center", fontsize=7.5, color=INK2)
ax.set_xlabel("tokens in book (log)"); ax.set_ylabel("books, shortest to longest")
ax.set_title(f"{(bc.tokens > 32768).sum()} of {len(bc)} books exceed 32K; "
             f"{(bc.tokens > 131072).sum()} exceed 128K",
             loc="left", fontsize=10, color=INK, pad=14)
ax.grid(color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
for lab, dx in [("les_miserables", -8), ("the_boxcar_children", 8)]:
    r = s[s["doc"].str.startswith(lab)]
    if len(r):
        ax.annotate(lab, (r["tokens"].iloc[0], r.index[0] + 1), fontsize=7,
                    color=INK2, textcoords="offset points", xytext=(dx, 6),
                    ha="right" if dx < 0 else "left")

# B - how far apart are consecutive mentions of the same character
ax = axes[0, 1]
gs  = np.sort(g); n = len(gs)
idx = np.unique(np.geomspace(1, n - 2, 500).astype(int))
ax.plot(gs[idx], 100 * (1 - idx / n), color="#2a78d6", linewidth=2, zorder=3)
ax.set_xscale("log"); ax.set_yscale("log")
for v, lab in WINDOWS:
    pctv = 100 * (g > v).mean()
    ax.axvline(v, color=BASE, linewidth=1, linestyle=(0, (4, 3)), zorder=1)
    if pctv > 0:
        ax.scatter([v], [pctv], s=34, color="#d03b3b", zorder=4,
                   edgecolor=SURFACE, linewidth=1)
        ax.annotate(f"{lab}: {pctv:.2f}%", (v, pctv), fontsize=7.5, color=INK2,
                    textcoords="offset points", xytext=(7, 5))
ax.set_xlabel("gap to previous mention (tokens, log)")
ax.set_ylabel("% of mentions with a larger gap")
ax.set_title("Long-range reference is rare - but not absent", loc="left",
             fontsize=10, color=INK, pad=14)
ax.grid(color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

# C - does the cast grow with the book?
ax = axes[1, 0]
ax.scatter(bc["tokens"], bc["chars"], s=34, color="#2a78d6", alpha=0.78,
           edgecolor=SURFACE, linewidth=0.9, zorder=3)
ax.set_xscale("log"); ax.set_xlabel("tokens in book (log)")
ax.set_ylabel("annotated characters")
ax.set_title(f"Cast size barely tracks length  (r = {bc['tokens'].corr(bc['chars']):+.2f})",
             loc="left", fontsize=10, color=INK, pad=14)
ax.grid(color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

# D - protagonist dominance
ax = axes[1, 1]
share = 100 * bc["biggest"] / bc["mentions"]
ax.hist(share, bins=np.arange(10, 75, 5), color="#2a78d6",
        edgecolor=SURFACE, linewidth=1.2, zorder=3)
ax.axvline(share.median(), color="#d03b3b", linewidth=1.6, zorder=4)
ax.annotate(f"median {share.median():.0f}%", (share.median(), ax.get_ylim()[1]*0.9),
            fontsize=7.5, color=INK2, textcoords="offset points", xytext=(7, 0))
ax.set_xlabel("% of a book's mentions held by its top character")
ax.set_ylabel("books")
ax.set_title("One character dominates each book", loc="left", fontsize=10, color=INK, pad=14)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

fig.suptitle("BookCoref - 53 full books, 11.1M tokens, 991,641 character mentions",
             x=0.008, ha="left", fontsize=12.5, color=INK)
fig.tight_layout(rect=[0, 0, 1, 0.955])
plt.show()

print(f"books over 32K tokens: {(bc.tokens > 32768).sum()}/{len(bc)}"
      f" | over 128K: {(bc.tokens > 131072).sum()}/{len(bc)}")
print(f"mentions needing >4K memory: {int((g > 4096).sum()):,}"
      f" | >32K: {int((g > 32768).sum()):,} | longest gap: {g.max():,} tokens")

---

# 4 · EvolvTrip / LitCharToM

20 books, 205 characters tracked across plot points, with belief / desire /
intention / emotion triples plus multiple-choice QA. This is the closest thing
in the corpus to ground-truth narrative state.

Needs `data/evolvtrip_data`.

In [ ]:
# ── BLOCK 13 ── EvolvTrip: structure + aggregates  (output truncated by design)
import json, collections
from pathlib import Path
import pandas as pd

ET = DATA / "evolvtrip_data"

def show(label, v, width=100):
    n = len(v) if hasattr(v, "__len__") else "-"
    print(f"  {label:22} {type(v).__name__:5} len={str(n):>6}  {repr(v)[:width]}")

for f in sorted(ET.glob("*.json")):
    print(f"{f.name:28} {f.stat().st_size/1e6:6.2f} MB")

for fname in ["all_books_current.json", "ood_test_book.json"]:
    d = json.load(open(ET / fname, encoding="utf-8"))
    print(f"\n=== {fname} - {len(d)} records ===")
    for k, v in d[0].items():
        show(k, v)
    # one level deeper on the nested fields
    for k in ["triples", "qa_data", "corresponding_triples", "messages"]:
        if k in d[0] and isinstance(d[0][k], list) and d[0][k]:
            print(f"  +- {k}[0]:")
            inner = d[0][k][0]
            if isinstance(inner, dict):
                for ik, iv in inner.items():
                    show("     " + ik, iv, 80)
            else:
                show("     [0]", inner, 120)

# --- aggregates ---
cur  = json.load(open(ET / "all_books_current.json", encoding="utf-8"))
prev = json.load(open(ET / "all_books_with_prev.json", encoding="utf-8"))
ood  = json.load(open(ET / "ood_test_book.json", encoding="utf-8"))

df_cur = pd.DataFrame([{"book": r["book_name"], "character": r["character"],
                        "plot_index": r["plot_index"],
                        "n_triples": len(r.get("triples") or []),
                        "n_qa": len(r.get("qa_data") or []),
                        "summary_words": len(str(r.get("plot_summary", "")).split()),
                        "scenario_words": len(str(r.get("scenario", "")).split())}
                       for r in cur])

print(f"\n=== all_books_current: {len(cur)} records ===")
print(f"  books      : {df_cur['book'].nunique()}")
print(f"  characters : {df_cur['character'].nunique()} distinct names,"
      f" {df_cur.groupby('book')['character'].nunique().mean():.1f} per book")
print(f"  plot_index : {df_cur.plot_index.min()}-{df_cur.plot_index.max()},"
      f" median {int(df_cur.groupby('book').plot_index.max().median())} plot points per book")
print(df_cur[["n_triples", "n_qa", "summary_words", "scenario_words"]].describe().round(1).to_string())

print("\n  records per book (top 8):")
print(df_cur.book.value_counts().head(8).to_string())

# NOTE: n_triples / n_qa read 1 because both fields are dicts with a single
# 'Target Character' key - the real triples live inside. Block 14 parses them.
dims = collections.Counter()
for r in cur:
    for t in (r.get("triples") or []):
        dims[t[1] if isinstance(t, (list, tuple)) and len(t) > 1 else str(t)[:40]] += 1
print("\n  triple relations (top 12):", dict(dims.most_common(12)))

print(f"\n=== ood_test_book: {len(ood)} records ===")
print("  qa_type   :", dict(collections.Counter(r.get("qa_type") for r in ood).most_common()))
print("  books     :", dict(collections.Counter(r.get("book_name") for r in ood).most_common(5)))
print("  answers   :", dict(collections.Counter(str(r.get("correct_answer"))[:20] for r in ood).most_common(6)))
print(f"\n  with_prev adds 'previous_plots': "
      f"{sum(1 for r in prev if r.get('previous_plots'))}/{len(prev)} records populated")

In [ ]:
# ── BLOCK 14 ── EvolvTrip: parse the triples, chart the structure
import re, collections
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

TRI = re.compile(r"^\(\s*([^,]+?)\s*,\s*([A-Za-z]+)\s*,\s*(.+?)\s*\)$")
DIM = [("Believes", "Belief", "#2a78d6"), ("Desires", "Desire", "#eb6834"),
       ("Intends", "Intention", "#1baf7a"), ("Feels", "Emotion", "#eda100")]

def dim_of(rel):
    for pre, name, _ in DIM:
        if rel.startswith(pre):
            return name
    return "other"

rels, per_rec, unparsed = collections.Counter(), [], []
for r in cur:
    strs = [s for v in (r["triples"] or {}).values() for s in v]
    per_rec.append(len(strs))
    for s in strs:
        m = TRI.match(s.strip())
        if m:
            rels[m.group(2)] += 1
        else:
            unparsed.append(s)

print(f"triples: {sum(per_rec):,} across {len(cur)} records"
      f" | median {int(np.median(per_rec))}/record | unparsed {len(unparsed)}")
if unparsed:
    print("  e.g.", unparsed[0][:110])
by_dim = collections.Counter(dim_of(r) for r in rels.elements())
print("dimensions:", dict(by_dim.most_common()))
print(f"distinct relations: {len(rels)} | top:", dict(rels.most_common(10)))

# qa_data inner shape + in-distribution answer key
qa_keys, in_ans = collections.Counter(), collections.Counter()
for r in cur:
    for lst in (r["qa_data"] or {}).values():
        for item in lst:
            for qtype, body in (item or {}).items():
                if isinstance(body, dict):
                    qa_keys.update(body.keys())
                    for k, v in body.items():
                        if "answer" in k.lower():
                            in_ans[str(v).strip()[:1]] += 1
print("\nqa_data inner keys:", dict(qa_keys.most_common(8)))
print("in-distribution answer key:", dict(in_ans.most_common()))

# ---------- charts ----------
fig, axes = plt.subplots(2, 2, figsize=(13, 8.2))

ax = axes[0, 0]
top = rels.most_common(14)[::-1]
cols = [dict((n, c) for _, n, c in DIM).get(dim_of(r), "#898781") for r, _ in top]
ax.barh(range(len(top)), [n for _, n in top], color=cols, height=0.72, zorder=3)
ax.set_yticks(range(len(top))); ax.set_yticklabels([r for r, _ in top], fontsize=7.5, color=INK2)
ax.set_xlabel("triples"); ax.set_title("Relation vocabulary  (top 14, coloured by dimension)",
                                       loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="x", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for _, n, c in DIM],
          labels=[n for _, n, _ in DIM], loc="lower right", frameon=False,
          fontsize=8, labelcolor=INK2)

ax = axes[0, 1]
letters = ["A", "B", "C", "D"]
ood_counts = [sum(1 for r in ood if r.get("correct_answer") == L) for L in letters]
tot = sum(ood_counts)
ax.bar(letters, [100*c/tot for c in ood_counts], color="#2a78d6", width=0.62, zorder=3)
ax.axhline(25, color="#d03b3b", linewidth=1.6, zorder=4)
ax.annotate("chance 25%", (3.4, 26), fontsize=7.5, color=INK2, ha="right")
for i, c in enumerate(ood_counts):
    ax.text(i, 100*c/tot + 1.2, f"{c}", ha="center", fontsize=8, color=INK2)
ax.set_ylabel("% of questions"); ax.set_xlabel("correct answer")
ax.set_title(f"OOD answer key is skewed - always-B scores {100*max(ood_counts)/tot:.0f}%",
             loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

ax = axes[1, 0]
pp = df_cur.groupby("book")["plot_index"].max().add(1).sort_values()
ax.barh(range(len(pp)), pp.values, color="#2a78d6", height=0.72, zorder=3)
ax.set_yticks(range(len(pp)))
ax.set_yticklabels([b[:32] for b in pp.index], fontsize=6.6, color=INK2)
ax.set_xlabel("plot points (timesteps) per book")
ax.set_title("How long is a character's tracked trajectory", loc="left",
             fontsize=10, color=INK, pad=8)
ax.grid(axis="x", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)

ax = axes[1, 1]
ax.hist(per_rec, bins=range(0, max(per_rec) + 2), color="#2a78d6",
        edgecolor=SURFACE, linewidth=1.2, zorder=3)
ax.set_xlabel("triples per (character, plot point)"); ax.set_ylabel("records")
ax.set_title("State density per timestep", loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

fig.suptitle("EvolvTrip / LitCharToM - 20 books, 205 characters, 638 character-timesteps",
             x=0.008, ha="left", fontsize=12.5, color=INK)
fig.tight_layout(rect=[0, 0, 1, 0.955])
plt.show()

In [ ]:
# ── BLOCK 15 ── EvolvTrip: collapse the relation vocabulary
# The 1,169 "distinct relations" are an artefact: the target character's name is
# concatenated onto the relation (FeelsTowardsSue, BelievesAboutToad). Strip it
# and the vocabulary collapses to ~10 base relations.
import collections
import matplotlib.pyplot as plt

BASES = ["FeelsTowards", "FeelsAbout", "BelievesAbout", "BelievesThat", "BelievesIn",
         "DesiresToKnow", "DesiresTo", "IntendsTo", "Intends",
         "Feels", "Believes", "Desires"]
BASES.sort(key=len, reverse=True)          # longest prefix wins

base_c, arg_c, raw_c = collections.Counter(), collections.Counter(), 0
for r in cur:
    for s in [x for v in (r["triples"] or {}).values() for x in v]:
        parts = s.strip().lstrip("(").rstrip(")").split(",", 2)
        if len(parts) < 3:
            continue
        raw_c += 1
        rel = parts[1].strip()
        for b in BASES:
            if rel.startswith(b):
                base_c[b] += 1
                if rel[len(b):].strip():
                    arg_c[b] += 1
                break
        else:
            base_c[f"?{rel[:24]}"] += 1

print(f"triples parsed: {raw_c:,}")
print(f"base relations: {len(base_c)}  vs {len(rels)} raw strings\n")
print(f"{'relation':16}{'count':>7}{'with object':>13}")
for b, n in base_c.most_common():
    print(f"{b:16}{n:>7}{arg_c.get(b, 0):>13}")

fig, ax = plt.subplots(figsize=(7.4, 4.2))
top = base_c.most_common(12)[::-1]
cols = [dict((p, c) for p, _, c in DIM).get(
            next((p for p, _, _ in DIM if b.startswith(p)), None), "#898781")
        for b, _ in top]
ax.barh(range(len(top)), [n for _, n in top], color=cols, height=0.72, zorder=3)
ax.set_yticks(range(len(top))); ax.set_yticklabels([b for b, _ in top], fontsize=8, color=INK2)
ax.set_xlabel("triples")
ax.set_title("Base relation vocabulary after stripping the object",
             loc="left", fontsize=10.5, color=INK, pad=8)
ax.grid(axis="x", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for _, _, c in DIM],
          labels=[n for _, n, _ in DIM], loc="lower right", frameon=False,
          fontsize=8, labelcolor=INK2)
fig.tight_layout(); plt.show()

---

# 5 · ConStory-Bench

2,000 prompts, 66,000 generated stories from 33 models, judged against a
taxonomy of **5 categories / 19 consistency subtypes**. This is the primary
consistency evaluation.

Needs `data/constory_bench` (including the LFS payload under `hf_data/`).

In [ ]:
# ── BLOCK 16 ── ConStory-Bench: structure + aggregates
from pathlib import Path
import collections
import pandas as pd
import pyarrow.parquet as pq

CB = DATA / "constory_bench"
HF = CB / "hf_data"

print("=== github repo (evaluation code) ===")
for p in sorted(CB.iterdir()):
    if p.name.startswith(".") or p.name == "hf_data":
        continue
    if p.is_dir():
        print(f"  {p.name:24} dir   files={sum(1 for _ in p.rglob('*') if _.is_file())}")
    else:
        print(f"  {p.name:24} {p.stat().st_size/1e3:9.1f} KB")

print("\n=== hf_data payload ===")
for p in sorted(HF.rglob("*")):
    if p.is_file() and ".git" not in p.parts:
        kb = p.stat().st_size / 1e3
        flag = "  <- LFS pointer, not fetched" if kb < 1 else ""
        print(f"  {str(p.relative_to(HF)):42} {kb/1e3:9.2f} MB{flag}")

pr = pd.read_parquet(HF / "prompts.parquet")
print(f"\n=== prompts.parquet {pr.shape} ===")
print(pr.dtypes.to_string())
print("\ntask_type x language:")
print(pd.crosstab(pr["task_type"], pr["language"]).to_string())

pr["words"] = pr["prompt"].astype(str).str.split().str.len()
print("\nprompt length in words:")
print(pr.groupby("task_type")["words"].describe().round(1).to_string())

print("\n=== one prompt per task_type (first 420 chars) ===")
for tt, g_ in pr.groupby("task_type"):
    print(f"\n--- {tt} ---\n{str(g_['prompt'].iloc[0])[:420]}")

for f in sorted(HF.glob("*.parquet")):
    if f.name == "prompts.parquet" or f.stat().st_size < 1000:
        continue
    pf = pq.ParquetFile(f)
    print(f"\n=== {f.name}: {pf.metadata.num_rows:,} rows x {pf.metadata.num_columns} cols ===")
    print("  cols:", pf.schema.names[:20])
    head = next(pf.iter_batches(batch_size=2)).to_pandas()
    for c in head.columns[:6]:
        print(f"   {c:22} {str(head[c].iloc[0])[:90]!r}")

ev = HF / "evaluations"
if ev.is_dir():
    files = sorted(p for p in ev.rglob("*") if p.is_file())
    print(f"\n=== evaluations/: {len(files)} files ===")
    print("  e.g.", [p.name for p in files[:5]])
    csvs = [p for p in files if p.suffix == ".csv" and p.stat().st_size > 1000]
    if csvs:
        d = pd.read_csv(csvs[0], nrows=3)
        print(f"  {csvs[0].name} cols: {list(d.columns)}")

In [ ]:
# ── BLOCK 17 ── ConStory-Bench: what does an evaluation row contain?
from pathlib import Path
import collections
import pandas as pd

EV = HF / "evaluations"
CATS = ["characterization", "factual_detail", "narrative_style", "timeline_plot", "world_building"]

# do all 33 model files share a schema?
headers = {}
for f in sorted(EV.glob("*.csv")):
    cols = tuple(pd.read_csv(f, nrows=0).columns)
    headers.setdefault(tuple(c for c in cols if not c.endswith("_story")), []).append(f.stem)
print(f"distinct schemas across {sum(len(v) for v in headers.values())} model files: {len(headers)}")
for cols, models in headers.items():
    print(f"  {len(models)} models | {len(cols)} shared cols | e.g. {models[:3]}")

f = EV / "claude_sonnet_45.csv"
d = pd.read_csv(f, nrows=3)
sub = [c for c in d.columns if any(c.startswith(k) for k in CATS)]
print(f"\n=== {f.name}: {len(d.columns)} cols, {len(sub)} subtype cols ===")
print("\nnon-subtype cols:", [c for c in d.columns if c not in sub])
print("\ndtypes of subtype cols:", dict(collections.Counter(str(d[c].dtype) for c in sub)))

print("\n=== first row, every subtype column (truncated) ===")
for c in sub:
    v = d[c].iloc[0]
    print(f"  {c:48} {type(v).__name__:7} {str(v)[:110]!r}")

print("\n=== how do 'story' and the model story column differ? ===")
for c in [x for x in d.columns if x.endswith("story") or x == "story"]:
    print(f"  {c:32} len={len(str(d[c].iloc[0])):7,}  {str(d[c].iloc[0])[:90]!r}")

# The "1-word" prompts are the Chinese ones - whitespace splitting does not work
# on Chinese text, so the word counts above are invalid for those 12 rows only.
print("\n=== shortest prompts by whitespace word count ===")
short = pr.nsmallest(3, "words")[["id", "task_type", "words", "prompt"]]
for _, r in short.iterrows():
    print(f"  id={r['id']} {r['task_type']:12} {r['words']:3}w  {str(r['prompt'])[:120]!r}")
print("\nlanguage counts:", dict(pr["language"].value_counts()))

> **Slow cell ahead.** The next cell reads **~2.7 GB across 33 CSVs** to count
> every judged finding. It took ~30 s on Colab and prints per-model progress.

In [ ]:
# ── BLOCK 18 ── ConStory-Bench: aggregate 33 models x 2,000 stories
import json, ast, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

CATS = {"characterization": "#2a78d6", "timeline_plot": "#eb6834",
        "factual_detail": "#1baf7a", "narrative_style": "#eda100",
        "world_building": "#e87ba4"}
base = list(pd.read_csv(EV / "claude_sonnet_45.csv", nrows=0).columns)
SUB  = [c for c in base if any(c.startswith(k) for k in CATS)]

for f in sorted(EV.glob("*.csv")):
    cols = set(pd.read_csv(f, nrows=0).columns)
    extra = cols - set(base) - {f"{f.stem}_story"}
    if extra:
        print(f"  {f.stem}: extra column(s) {sorted(extra)}")

def n_find(cell):
    if not isinstance(cell, str):
        return 0
    s = cell.strip()
    if s in ("", "[]", "nan"):
        return 0
    try:
        v = json.loads(s)
    except Exception:
        try:
            v = ast.literal_eval(s)
        except Exception:
            return 0
    return len(v) if isinstance(v, list) else 0

long, denom, t0 = [], [], time.time()
for i, f in enumerate(sorted(EV.glob("*.csv")), 1):
    cols = set(pd.read_csv(f, nrows=0).columns)
    use  = ["task_type"] + [c for c in SUB if c in cols]
    d    = pd.read_csv(f, usecols=use)
    for c in use[1:]:
        d[c] = d[c].map(n_find)
    g_ = d.groupby("task_type")[use[1:]].sum()
    for tt, row in g_.iterrows():
        for c in use[1:]:
            long.append((f.stem, tt, c, int(row[c])))
    for tt, n in d.groupby("task_type").size().items():
        denom.append((f.stem, tt, int(n)))
    print(f"  [{i:2}/33] {f.stem:32} rows={len(d):5}  findings={int(d[use[1:]].values.sum()):6,}"
          f"  {time.time()-t0:5.0f}s")

L = pd.DataFrame(long, columns=["model", "task_type", "subtype", "n"])
D = pd.DataFrame(denom, columns=["model", "task_type", "stories"])
L["category"] = L["subtype"].str.extract(f"^({'|'.join(CATS)})")
print(f"\ntotal findings: {L['n'].sum():,} across {D['stories'].sum():,} stories")

per_model = (L.groupby("model")["n"].sum() /
             D.groupby("model")["stories"].sum()).sort_values()

# ---------- charts ----------
fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.4), gridspec_kw={"height_ratios": [1.45, 1]})

ax = axes[0, 0]
ax.barh(range(len(per_model)), per_model.values, color="#2a78d6", height=0.74, zorder=3)
ax.set_yticks(range(len(per_model)))
ax.set_yticklabels(per_model.index, fontsize=6.4, color=INK2)
ax.set_xlabel("contradictions per story")
ax.set_title("Consistency leaderboard  (lower is better)", loc="left",
             fontsize=10, color=INK, pad=8)
ax.grid(axis="x", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)

ax = axes[0, 1]
sub_tot = L.groupby(["category", "subtype"])["n"].sum().sort_values()
cols = [CATS[c] for c, _ in sub_tot.index]
ax.barh(range(len(sub_tot)), sub_tot.values, color=cols, height=0.74, zorder=3)
ax.set_yticks(range(len(sub_tot)))
ax.set_yticklabels([s.replace(f"{c}_", "") for c, s in sub_tot.index], fontsize=7, color=INK2)
ax.set_xlabel("findings across all 66,000 stories")
ax.set_title("What actually breaks in long-form generation", loc="left",
             fontsize=10, color=INK, pad=8)
ax.grid(axis="x", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in CATS.values()],
          labels=list(CATS), loc="lower right", frameon=False, fontsize=7.5, labelcolor=INK2)

ax = axes[1, 0]
tt_rate = (L.groupby("task_type")["n"].sum() / D.groupby("task_type")["stories"].sum())
tt_rate = tt_rate.sort_values()
ax.bar(range(len(tt_rate)), tt_rate.values, color="#2a78d6", width=0.62, zorder=3)
ax.set_xticks(range(len(tt_rate))); ax.set_xticklabels(tt_rate.index, fontsize=8.5, color=INK2)
for i, v in enumerate(tt_rate.values):
    ax.text(i, v * 1.02, f"{v:.2f}", ha="center", fontsize=8, color=INK2)
ax.set_ylabel("contradictions per story")
ax.set_title("Task type barely moves the rate", loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

ax = axes[1, 1]
cat_share = L.groupby("category")["n"].sum().sort_values()
ax.barh(range(len(cat_share)), cat_share.values,
        color=[CATS[c] for c in cat_share.index], height=0.68, zorder=3)
ax.set_yticks(range(len(cat_share))); ax.set_yticklabels(cat_share.index, fontsize=8.5, color=INK2)
tot = cat_share.sum()
for i, v in enumerate(cat_share.values):
    ax.text(v * 1.01, i, f"{100*v/tot:.0f}%", va="center", fontsize=8, color=INK2)
ax.set_xlabel("findings"); ax.set_xlim(0, cat_share.max() * 1.12)
ax.set_title("Category share", loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="x", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)

fig.suptitle("ConStory-Bench - 33 models x 2,000 prompts, 19 consistency subtypes",
             x=0.008, ha="left", fontsize=12.5, color=INK)
fig.tight_layout(rect=[0, 0, 1, 0.955])
plt.show()

print("\nbest 5:\n" + per_model.head(5).round(2).to_string())
print("\nworst 5:\n" + per_model.tail(5).round(2).to_string())

---

# 6 · FABLES

26 books x 5 summarizers = 3,158 claim-level faithfulness judgements, each with
evidence and (usually) a written reason.

Needs `data/fables`. The first cell unzips `FABLES.json` if it isn't extracted yet.

> **Fixed from the live session:** `claims` is a **dict keyed by string indices**
> (`'0'`, `'1'`, …), not a list, so the original `claims[0]` probe raised
> `KeyError: 0`. Block 19 below stops at the leaf-key probe and Block 20 does the
> claim work with the correct dict handling.

In [ ]:
# ── BLOCK 19 ── FABLES: load + top-level structure
import zipfile, json, collections
from pathlib import Path
import pandas as pd

FB = DATA / "fables"
jf = FB / "data" / "FABLES.json"
if not jf.exists():
    with zipfile.ZipFile(FB / "data" / "FABLES.json.zip") as z:
        z.extractall(FB / "data")
    print("extracted FABLES.json")

raw = json.load(open(jf, encoding="utf-8"))
print("top-level keys:", list(raw))
print("canary present:", "canary" in raw, "| length:", len(str(raw.get("canary", ""))))

fab   = raw["FABLES"]
books = list(fab)
summ  = sorted({s for b in fab.values() for s in b})
print(f"\nbooks: {len(books)} | summarizers: {summ}")
print("book titles:", [b[:34] for b in books[:6]], "...")

leaf = fab[books[0]][summ[0]]
print(f"\nleaf keys for ({books[0][:28]}, {summ[0]}):")
for k, v in leaf.items():
    n = len(v) if hasattr(v, "__len__") else "-"
    print(f"  {k:18} {type(v).__name__:5} len={str(n):>6}  {str(v)[:90]!r}")

In [ ]:
# ── BLOCK 20 ── FABLES: claims are a dict, not a list
import collections
import pandas as pd

claims = leaf["claims"]
k0 = next(iter(claims))
print(f"claims: {type(claims).__name__} with {len(claims)} entries | first key {k0!r}")
print(f"\nclaims[{k0!r}]:")
for k, v in claims[k0].items():
    n = len(v) if hasattr(v, "__len__") else "-"
    print(f"  {k:22} {type(v).__name__:5} len={str(n):>5}  {str(v)[:110]!r}")

# --- flatten every claim across every book x summarizer ---
rows, nested, missing = [], collections.Counter(), collections.Counter()
for b, per_sum in fab.items():
    for s, entry in per_sum.items():
        cl = entry.get("claims") or {}
        items = cl.items() if isinstance(cl, dict) else enumerate(cl)
        for cid, c in items:
            rec = {"book": b[:34], "summarizer": s, "claim_id": str(cid)}
            if isinstance(c, dict):
                for k, v in c.items():
                    if isinstance(v, (list, dict)):
                        nested[k] += 1
                    else:
                        rec[k] = v
            else:
                rec["claim"] = str(c)
            rows.append(rec)
        if not cl:
            missing[s] += 1

C = pd.DataFrame(rows)
print(f"\n=== {len(C):,} claims | columns: {list(C.columns)} ===")
if nested:
    print("nested fields skipped (they get their own pass in Block 21):", dict(nested))
if missing:
    print("book x summarizer cells with no claims:", dict(missing))

for col in C.columns:
    if col in ("book", "summarizer", "claim_id"):
        continue
    u = C[col].nunique(dropna=False)
    if u <= 15:
        print(f"\n{col} ({u} values):")
        print(C[col].value_counts(dropna=False).to_string())
    else:
        w = C[col].astype(str).str.split().str.len()
        print(f"\n{col}: {u:,} distinct | words median {w.median():.0f}, max {w.max()}")

print("\nclaims per summarizer:")
print(C.groupby("summarizer").size().to_string())
print("\nclaims per book x summarizer - is the grid complete?")
grid = C.pivot_table(index="book", columns="summarizer", values="claim_id", aggfunc="count")
print(f"  cells filled: {grid.notna().sum().sum()}/{grid.shape[0]*grid.shape[1]}"
      f" | claims per cell: min {grid.min().min():.0f}, median {grid.stack().median():.0f}, max {grid.max().max():.0f}")
print(grid.head(6).to_string())

In [ ]:
# ── BLOCK 21 ── FABLES: the nested pass + charts
import collections
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

LBL = {"Yes": "#0ca30c", "PartialSupport": "#fab219",
       "No": "#d03b3b", "Inapplicable": "#898781"}
ORDER_L = list(LBL)

rows = []
for b, per_sum in fab.items():
    for s, entry in per_sum.items():
        for cid, c in (entry.get("claims") or {}).items():
            ev = [str(e) for e in (c.get("evidence") or []) if str(e).strip()]
            rs = [str(r) for r in (c.get("reason") or []) if str(r).strip()]
            rows.append({"book": b[:34], "summarizer": s, "label": c.get("label"),
                         "claim_words": len(str(c.get("claim", "")).split()),
                         "n_evidence": len(ev),
                         "ev_words": sum(len(e.split()) for e in ev),
                         "has_reason": int(bool(rs)),
                         "reason_words": sum(len(r.split()) for r in rs)})
F = pd.DataFrame(rows)

print(f"claims: {len(F):,}")
print("\nevidence per claim by label:")
print(F.groupby("label")[["n_evidence", "ev_words"]].mean().round(2).to_string())
print("\nreason given, by label:")
r = F.groupby("label")["has_reason"].agg(["sum", "mean", "count"])
r["mean"] = (100 * r["mean"]).round(1)
print(r.rename(columns={"sum": "with_reason", "mean": "pct", "count": "claims"}).to_string())
print(f"\nreason word count where present: median "
      f"{F.loc[F.has_reason == 1, 'reason_words'].median():.0f}")

fig, axes = plt.subplots(2, 2, figsize=(13, 8.6), gridspec_kw={"height_ratios": [1, 1.25]})

# A - label mix per summarizer
ax = axes[0, 0]
piv = (F.pivot_table(index="summarizer", columns="label", values="book", aggfunc="count")
         .reindex(columns=ORDER_L).fillna(0))
piv = piv.loc[(piv["No"] + piv["PartialSupport"]).div(piv.sum(axis=1)).sort_values().index]
pct = 100 * piv.div(piv.sum(axis=1), axis=0)
left = np.zeros(len(piv))
for lab in ORDER_L:
    v = pct[lab].values
    ax.barh(range(len(piv)), v, left=left, color=LBL[lab], height=0.66,
            edgecolor=SURFACE, linewidth=1.4, label=lab, zorder=3)
    for i, (x, l) in enumerate(zip(v, left)):
        if x >= 6:
            ax.text(l + x/2, i, f"{x:.0f}", ha="center", va="center",
                    fontsize=7.5, color=ink_on(LBL[lab]))
    left += v
ax.set_yticks(range(len(piv)))
ax.set_yticklabels([f"{s}\n{int(n):,} claims" for s, n in zip(piv.index, piv.sum(axis=1))],
                   fontsize=7.5, color=INK2)
ax.set_xlim(0, 100); ax.set_xticks([0, 50, 100]); ax.set_xticklabels(["0", "50", "100%"])
ax.set_title("Claim verification by summarizer  (%)", loc="left", fontsize=10, color=INK, pad=8)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.26), ncol=4, frameon=False,
          fontsize=8, labelcolor=INK2)

# B - evidence attached per label
ax = axes[0, 1]
m = F.groupby("label")[["n_evidence"]].mean().reindex(ORDER_L)
ax.bar(range(len(m)), m["n_evidence"].values,
       color=[LBL[l] for l in m.index], width=0.62, zorder=3)
ax.set_xticks(range(len(m))); ax.set_xticklabels(m.index, fontsize=8, color=INK2)
for i, v in enumerate(m["n_evidence"].values):
    ax.text(i, v * 1.02, f"{v:.2f}", ha="center", fontsize=8, color=INK2)
ax.set_ylabel("evidence snippets per claim")
ax.set_title("Does a failing claim carry less evidence?", loc="left",
             fontsize=10, color=INK, pad=8)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

# C - hardest books  (pooled across all five summarizers)
ax = axes[1, 0]
bk = F[F.label != "Inapplicable"].groupby("book")["label"].agg(
    lambda s: 100 * (s != "Yes").mean()).sort_values()
ax.barh(range(len(bk)), bk.values, color="#d03b3b", height=0.72, zorder=3)
ax.set_yticks(range(len(bk))); ax.set_yticklabels(bk.index, fontsize=6.4, color=INK2)
ax.axvline(bk.mean(), color=BASE, linewidth=1.2, linestyle=(0, (4, 3)), zorder=4)
ax.set_xlabel("% of applicable claims not fully supported")
ax.set_title("Which books defeat summarizers  (all 5 pooled)", loc="left",
             fontsize=10, color=INK, pad=8)
ax.grid(axis="x", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)

# D - claims per cell
ax = axes[1, 1]
cell = F.groupby(["book", "summarizer"]).size()
ax.hist(cell.values, bins=range(10, 46, 2), color="#2a78d6",
        edgecolor=SURFACE, linewidth=1.2, zorder=3)
ax.set_xlabel("claims extracted from one summary"); ax.set_ylabel("book x summarizer cells")
ax.set_title(f"Summary granularity varies  (median {cell.median():.0f})",
             loc="left", fontsize=10, color=INK, pad=8)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

fig.suptitle("FABLES - 26 books x 5 summarizers, 3,158 verified claims",
             x=0.008, ha="left", fontsize=12.5, color=INK)
fig.tight_layout(rect=[0, 0, 1, 0.955])
plt.show()

---

# 7 · GPT4-Books

571 books x 100 name-cloze instances x 2 model runs. Accuracy here is a
memorization signal: the high-scoring tail is a contamination blacklist.

Needs `data/gpt4_books`. The last cell reads 1,142 files.

In [ ]:
# ── BLOCK 22 ── GPT4-Books: structure probe
from pathlib import Path
import collections, itertools

GB = DATA / "gpt4_books"

print("=== tree (dirs, <=3 levels) ===")
for p in sorted(GB.rglob("*")):
    if not p.is_dir() or ".git" in p.parts:
        continue
    rel = p.relative_to(GB)
    if len(rel.parts) <= 3:
        files = [f for f in p.iterdir() if f.is_file()]
        ext = collections.Counter(f.suffix or "<none>" for f in files)
        print(f"  {'  ' * (len(rel.parts)-1)}{rel.name:34} files={len(files):5} {dict(ext)}")

hits = [p for p in GB.rglob("chatgpt_results") if p.is_dir()]
print("\nchatgpt_results at:", [str(h.relative_to(GB)) for h in hits])
res = hits[0]
print("sibling result dirs:", sorted(d.name for d in res.parent.iterdir() if d.is_dir()))

files = sorted(res.glob("*.txt"))
print(f"\n{len(files)} books | e.g. {[f.stem for f in files[:4]]}")

f = files[0]
print(f"\n=== {f.name} - first 12 lines ===")
with open(f, encoding="utf-8") as fh:
    for i, line in enumerate(itertools.islice(fh, 12)):
        print(f"  {i:2} {line.rstrip()[:150]!r}")

n_lines = [sum(1 for _ in open(x, encoding="utf-8")) for x in files]
print(f"\nlines per file: min {min(n_lines)}, median {sorted(n_lines)[len(n_lines)//2]}, max {max(n_lines)}")
print(f"identical line counts: {len(set(n_lines))} distinct values -> {collections.Counter(n_lines).most_common(4)}")

print("\n=== does the repo ship the source passages too? ===")
for d in sorted(res.parent.parent.iterdir()):
    if d.is_dir():
        n = sum(1 for _ in d.rglob('*') if _.is_file())
        print(f"  {d.name:28} files={n}")

In [ ]:
# ── BLOCK 23 ── GPT4-Books: name-cloze accuracy as a memorization signal
# Row format: <name>PRED</name> \t PRED \t GOLD \t passage-with-[MASK]
# Note: only rows with >=3 tab-separated fields are counted, so the usable row
# count per book runs 57-100, not a flat 100.
import re, collections
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT_GB = res.parent                   # data/model_output
RUNS = {"chatgpt": ROOT_GB / "chatgpt_results", "gpt4": ROOT_GB / "gpt4_results"}
TITLE = re.compile(r"^(mr|mrs|miss|ms|dr|sir|lady|lord|master)\.?\s+")

def nm(s):
    return re.sub(r"[^a-z]", "", TITLE.sub("", s.strip().lower()))

def rows_of_book(p):
    out = []
    with open(p, encoding="utf-8") as f:
        for line in f:
            c = line.rstrip("\n").split("\t")
            if len(c) >= 3:
                out.append((c[0], c[1].strip(), c[2].strip()))
    return out

recs, malformed, golds = [], 0, collections.Counter()
for run, d in RUNS.items():
    for p in sorted(d.glob("*.txt")):
        r = rows_of_book(p)
        malformed += sum(1 for tag, pred, _ in r if tag != f"<name>{pred}</name>")
        ex  = np.mean([pred == gold for _, pred, gold in r])
        len_ = np.mean([nm(pred) == nm(gold) for _, pred, gold in r])
        if run == "gpt4":
            golds.update(gold for _, _, gold in r)
        recs.append({"run": run, "book": p.stem, "n": len(r),
                     "exact": 100 * ex, "lenient": 100 * len_})

G = pd.DataFrame(recs)
print(f"{len(G):,} book-runs | rows/book: {G.n.min()}-{G.n.max()}"
      f" | tag/pred mismatches: {malformed}")
print("\naccuracy across 571 books (%):")
print(G.groupby("run")[["exact", "lenient"]].describe().round(2).T.to_string())

W = G.pivot(index="book", columns="run", values="lenient")
print(f"\ncorrelation chatgpt vs gpt4: r = {W['chatgpt'].corr(W['gpt4']):+.2f}")
print(f"books at 0% (gpt4): {(W['gpt4'] == 0).sum()} / {len(W)}"
      f" | above 20%: {(W['gpt4'] > 20).sum()} | above 50%: {(W['gpt4'] > 50).sum()}")
print("\nmost memorized (gpt4, lenient %):")
print(W.sort_values("gpt4", ascending=False).head(10).round(1).to_string())

fig, axes = plt.subplots(2, 2, figsize=(13, 8.4))

ax = axes[0, 0]
bins = np.arange(0, 102, 4)
for run, col in [("chatgpt", "#2a78d6"), ("gpt4", "#eb6834")]:
    ax.hist(G[G.run == run]["lenient"], bins=bins, color=col, alpha=0.75,
            edgecolor=SURFACE, linewidth=1.0, label=run, zorder=3)
ax.set_xlabel("name-cloze accuracy (%)"); ax.set_ylabel("books")
ax.set_title("Most books are not memorized", loc="left", fontsize=10, color=INK, pad=8)
ax.legend(frameon=False, fontsize=8, labelcolor=INK2)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

ax = axes[0, 1]
top = W.sort_values("gpt4").tail(18)
y = np.arange(len(top))
ax.barh(y - 0.2, top["chatgpt"], height=0.38, color="#2a78d6", label="chatgpt", zorder=3)
ax.barh(y + 0.2, top["gpt4"], height=0.38, color="#eb6834", label="gpt4", zorder=3)
ax.set_yticks(y); ax.set_yticklabels([b[:30] for b in top.index], fontsize=6.4, color=INK2)
ax.set_xlabel("accuracy (%)")
ax.set_title("Most-memorized books", loc="left", fontsize=10, color=INK, pad=8)
ax.legend(frameon=False, fontsize=8, labelcolor=INK2, loc="lower right")
ax.grid(axis="x", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)
ax.spines["left"].set_visible(False); ax.tick_params(left=False)

ax = axes[1, 0]
ax.scatter(W["chatgpt"], W["gpt4"], s=22, color="#2a78d6", alpha=0.6,
           edgecolor=SURFACE, linewidth=0.6, zorder=3)
lim = max(W.max()) * 1.05
ax.plot([0, lim], [0, lim], color=BASE, linewidth=1, linestyle=(0, (4, 3)), zorder=2)
ax.set_xlim(-1, lim); ax.set_ylim(-1, lim)
ax.set_xlabel("chatgpt accuracy (%)"); ax.set_ylabel("gpt4 accuracy (%)")
ax.set_title("The two models memorize the same books", loc="left", fontsize=10, color=INK, pad=8)
ax.grid(color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

ax = axes[1, 1]
ex_len = G.groupby("run")[["exact", "lenient"]].mean()
x = np.arange(len(ex_len))
ax.bar(x - 0.19, ex_len["exact"], width=0.36, color="#2a78d6", label="exact match", zorder=3)
ax.bar(x + 0.19, ex_len["lenient"], width=0.36, color="#eb6834", label="titles/case stripped", zorder=3)
for i, (a, b) in enumerate(zip(ex_len["exact"], ex_len["lenient"])):
    ax.text(i - 0.19, a + 0.15, f"{a:.1f}", ha="center", fontsize=8, color=INK2)
    ax.text(i + 0.19, b + 0.15, f"{b:.1f}", ha="center", fontsize=8, color=INK2)
ax.set_xticks(x); ax.set_xticklabels(ex_len.index, fontsize=9, color=INK2)
ax.set_ylabel("mean accuracy (%)")
ax.set_title("Matching rule barely moves the number", loc="left", fontsize=10, color=INK, pad=8)
ax.legend(frameon=False, fontsize=8, labelcolor=INK2)
ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0); ax.set_axisbelow(True)

fig.suptitle("GPT4-Books - 571 books x 100 name-cloze instances x 2 models",
             x=0.008, ha="left", fontsize=12.5, color=INK)
fig.tight_layout(rect=[0, 0, 1, 0.955])
plt.show()